# BTC Price Predictor — `pridictior_snow9` (Snowflake Notebook)

Runs the **full pipeline offline** on the Snowflake container runtime
(4 × A10 23 GB, 48 vCPU, 100 GB RAM):

1. reads the dataset from the internal stage `@SNOW9.PUBLIC.BTC_SNOW9_STAGE`
   (**no internet required** — only PyPI for packages, which is allowed)
2. builds the Market-Analyser feature cache (CPU)
3. trains **four Price-Predictor variants, one per GPU**, with the
   streaming *real-time reward* (no epochs, 25-min-ahead, updated every second)
4. replays a **random market day (2020 → today)**, rendering the 30 fps
   animation: actual price, dotted 25-min forecasts per model, live accuracy bars


In [ ]:
# ---- 1. packages (PyPI is reachable; everything else is offline) ----
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "torch", "--index-url", "https://download.pytorch.org/whl/cu121"])
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "numpy", "pandas", "pyarrow", "matplotlib",
                "pyyaml", "tqdm", "imageio[ffmpeg]"])
print("packages ok")


In [ ]:
# ---- 2. pull code + data from the internal stage (NO internet) ----
import os, zipfile
from snowflake.snowpark.context import get_active_session
session = get_active_session()

REPO   = "/tmp/snow9/repo"
DATA   = "/tmp/snow9/data/raw"
os.makedirs(DATA, exist_ok=True)

files = session.sql("LIST @SNOW9.PUBLIC.BTC_SNOW9_STAGE").to_pandas()
display(files)

for row in files.itertuples():
    stage_path = row.name if row.name.startswith("@") else "@" + row.name
    if row.name.endswith(".zip"):
        session.file.get(stage_path, "/tmp/snow9/")
    elif row.name.endswith(".parquet"):
        session.file.get(stage_path, DATA + "/")

zipfile.ZipFile("/tmp/snow9/pridictior_snow9.zip").extractall("/tmp/snow9")
print("repo:", os.listdir(REPO)); print("data shards:", len(os.listdir(DATA)))


In [ ]:
# ---- 3. feature cache (single vectorised pass over every shard) ----
import sys, os
sys.path.insert(0, "/tmp/snow9/repo/src"); os.chdir("/tmp/snow9/repo")
from snow9.data import ShardStore, build_feature_cache
from snow9.models import MarketAnalyzer
meta = build_feature_cache(ShardStore("/tmp/snow9/data/raw"),
                           "/tmp/snow9/data/cache", MarketAnalyzer())
meta


In [ ]:
# ---- 4. TRAIN: 4 variants x 1 GPU each, streaming real-time reward ----
import torch
assert torch.cuda.device_count() == 4, "expected the 4 x A10 node"
for i in range(4): print(i, torch.cuda.get_device_name(i))

import subprocess, os
env = dict(os.environ, PYTHONPATH="/tmp/snow9/repo/src", OMP_NUM_THREADS="10")
subprocess.run(
    "torchrun --standalone --nproc_per_node=4 "
    "-m snow9.training.ddp_launch "
    "--data-cache /tmp/snow9/data/cache "
    "--variants configs/variants.yaml "
    "--training configs/training.yaml "
    "--out /tmp/snow9/checkpoints",
    shell=True, cwd="/tmp/snow9/repo", env=env, check=True)
# full snapshots every 30 market-minutes per variant — storage is not a constraint


In [ ]:
# ---- 5. REPLAY a random market day through ALL trained variants ----
import subprocess, os, sys
env = dict(os.environ, PYTHONPATH="/tmp/snow9/repo/src")
subprocess.run([sys.executable, "-m", "snow9.viz.rollout",
                "--cache", "/tmp/snow9/data/cache",
                "--ckpt-root", "/tmp/snow9/checkpoints",
                "--date", "random", "--duration-sec", "86400",
                "--out", "/tmp/snow9/runs/rollout_random.npz"],
               cwd="/tmp/snow9/repo", env=env, check=True)


In [ ]:
# ---- 6. RENDER the 30 fps animation and watch it here ----
import subprocess, os, sys
env = dict(os.environ, PYTHONPATH="/tmp/snow9/repo/src")
subprocess.run([sys.executable, "-m", "snow9.viz.animate",
                "/tmp/snow9/runs/rollout_random.npz",
                "--out", "/tmp/snow9/videos/predictions_random.mp4",
                "--fps", "30", "--seconds-per-frame", "120"],
               cwd="/tmp/snow9/repo", env=env, check=True)
from IPython.display import Video
Video("/tmp/snow9/videos/predictions_random.mp4", embed=True, width=1000)


### Notes
* Checkpoints (model + optimiser + EMA + RNG, every 30 market-min per variant) live in
  `/tmp/snow9/checkpoints/<variant>/` — persist them back to the stage with
  `session.file.put` if you want them to outlive this warehouse.
* Training resumes automatically from each variant's `latest.pt`.
* Not financial advice — see `README.md`.
